# 🤖 Claude Code — Guide Pratique pour Débutants

> **Basé sur** : [claude-code-best-practice](https://github.com/shanraisshan/claude-code-best-practice) par Shayan Raisshan  
> **Mis à jour** : Mai 2026 · Claude Code v2.1.150

---

Ce notebook couvre les concepts essentiels pour bien utiliser **Claude Code** au quotidien.  
Vous allez apprendre à organiser vos projets, créer des commandes réutilisables, configurer des sous-agents et bien plus.

## Plan du notebook

| # | Section | Concept clé |
|---|---------|------------|
| 1 | Installation & premiers pas | `claude`, `/doctor`, `/help` |
| 2 | La mémoire : CLAUDE.md | Contexte persistant |
| 3 | Les Commandes (Commands) | Slash commands personnalisées |
| 4 | Les Skills | Capacités réutilisables |
| 5 | Les Sous-agents (Subagents) | Délégation de tâches |
| 6 | Les Settings | Configuration avancée |
| 7 | Gestion du contexte | Éviter la "context rot" |
| 8 | Workflows de développement | Research → Plan → Execute → Review |
| 9 | Tips & Tricks (83 conseils) | Bonnes pratiques des créateurs |
| 10 | Architecture Command→Agent→Skill | Orchestration complète |

---
## Section 1 — Installation & Premiers Pas

### Qu'est-ce que Claude Code ?

Claude Code est un **agent de développement en ligne de commande** créé par Anthropic.  
Il tourne dans votre terminal et peut lire, écrire, éditer des fichiers, exécuter des commandes bash, et bien plus — en autonomie ou avec votre validation.

```
Votre terminal
    └── claude          ← le CLI
          ├── Lit vos fichiers
          ├── Écrit / modifie du code
          ├── Lance des commandes bash
          └── Appelle des sous-agents
```

### Installation

```bash
# Prérequis : Node.js 18+
npm install -g @anthropic-ai/claude-code

# Vérifier l'installation
claude --version

# Lancer Claude Code dans un projet
cd mon-projet/
claude
```

### Commandes de diagnostic essentielles

| Commande | Rôle |
|----------|------|
| `/doctor` | Diagnostique l'installation, l'auth, la config |
| `/help` | Liste toutes les commandes disponibles |
| `/status` | Affiche la version, le modèle, le compte |
| `/usage` | Coût de la session, limites du plan |

> 💡 **Tip du créateur (Boris Cherny)** : Mettez à jour Claude Code **chaque jour** (`npm update -g @anthropic-ai/claude-code`) — les nouvelles fonctionnalités arrivent très vite.

### 📁 Où se trouve le dossier `.claude` ?

Claude Code utilise **deux emplacements** pour stocker sa configuration :

#### 1. Dossier global (votre compte utilisateur)
```
Windows  : C:\Users\<votre_nom>\.claude\
Mac/Linux: ~/.claude/
```
Ce dossier contient votre configuration personnelle partagée entre tous vos projets :
```
~/.claude/
├── settings.json        ← configuration globale personnelle
├── CLAUDE.md            ← instructions globales (tous projets)
├── commands/            ← commandes slash globales
├── projects/            ← historique des sessions par projet
├── sessions/            ← sessions sauvegardées (/resume)
└── history.jsonl        ← historique des conversations
```

#### 2. Dossier projet (dans votre dépôt)
```
mon-projet/
└── .claude/             ← config spécifique à CE projet (versionné git)
    ├── settings.json    ← configuration de l'équipe
    ├── commands/        ← commandes slash du projet
    ├── agents/          ← sous-agents du projet
    ├── skills/          ← skills du projet
    └── hooks/           ← scripts de hooks
```

#### Hiérarchie de priorité

```
Config IT (managed)      ← priorité maximale
CLI flags (--model ...)
.claude/settings.local.json  ← personnel, NON versionné (dans .gitignore)
.claude/settings.json        ← équipe, versionné git
~/.claude/settings.json      ← personnel global
```

> ⚠️ **Attention** : `.claude/settings.local.json` est pour vos préférences personnelles (ex: votre modèle favori). Ne le commitez jamais — ajoutez-le à `.gitignore`.

In [ ]:
# Exercice 1 : Vérifier votre installation Claude Code
# Exécutez ces commandes dans votre terminal (pas ici dans Python)

installation_checklist = {
    "Node.js installé (v18+)": "node --version",
    "Claude Code installé": "claude --version",
    "Authentifié": "claude /status",
    "Diagnostic OK": "claude /doctor",
}

print("=== Checklist d'installation ===\n")
for etape, commande in installation_checklist.items():
    print(f"  ✅ {etape}")
    print(f"     → Commande : {commande}\n")

---
## Section 2 — La Mémoire : CLAUDE.md

### Concept

`CLAUDE.md` est le fichier de **mémoire persistante** de Claude Code.  
C'est le moyen le plus impactant d'améliorer la qualité des réponses pour votre projet.  
Claude le lit **automatiquement** à chaque démarrage de session.

### Où placer le fichier ?

```
mon-projet/
├── CLAUDE.md              ← Instructions pour CE projet
├── .claude/
│   └── rules/             ← Règles additionnelles (chargement lazy)
│       ├── testing.md
│       └── style.md
frontend/
├── CLAUDE.md              ← Instructions spécifiques au frontend
backend/
└── CLAUDE.md              ← Instructions spécifiques au backend
```

Il existe aussi un fichier global :
- **Mac/Linux** : `~/.claude/CLAUDE.md`
- **Windows** : `C:\Users\<vous>\.claude\CLAUDE.md`

### Mécanisme de chargement (monorepos)

| Situation | Chargé ? |
|-----------|---------|
| CLAUDE.md dans le dossier courant | ✅ Immédiatement |
| CLAUDE.md dans les dossiers parents | ✅ Immédiatement |
| CLAUDE.md dans les sous-dossiers | ⏳ Lazy (quand Claude touche ce dossier) |
| CLAUDE.md d'un dossier frère | ❌ Jamais |

### Que mettre dans CLAUDE.md ?

```markdown
# Mon Projet

## Stack technique
- Python 3.11, FastAPI, PostgreSQL
- Tests : pytest
- Linter : ruff

## Commandes importantes
- Lancer les tests : `pytest tests/`
- Démarrer le serveur : `uvicorn main:app --reload`
- Linter : `ruff check .`

## Conventions
- Branches git : feature/xxx, fix/xxx
- Commits : conventional commits (feat:, fix:, docs:)
- PR : toujours squash merge

## Ce qu'il NE faut PAS faire
- Ne jamais modifier la base de données en production directement
- Ne jamais commiter les fichiers .env
```

### Règles d'or pour CLAUDE.md

1. **Moins de 200 lignes** par fichier (au-delà, Claude commence à ignorer)
2. **Une tâche = une nouvelle session** — ne pas laisser dériver
3. **Commandes de test présentes** — n'importe qui doit pouvoir taper "run tests" et ça marche
4. **Balises `<important if="...">` pour les règles critiques** — évite qu'elles soient ignorées

> 💡 **Tip** : Utilisez `.claude/rules/*.md` pour séparer les grandes instructions en fichiers thématiques. Ajoutez un frontmatter `paths:` pour un chargement conditionnel.

In [ ]:
import os

# Exercice 2 : Générer un CLAUDE.md adapté à votre projet
def generer_claude_md(
    nom_projet: str,
    stack: list[str],
    commande_tests: str,
    commande_lancer: str,
    conventions: list[str]
) -> str:
    """Génère un template CLAUDE.md prêt à l'emploi."""
    
    stack_str = "\n".join(f"- {s}" for s in stack)
    conventions_str = "\n".join(f"- {c}" for c in conventions)
    
    return f"""# {nom_projet}

## Stack technique
{stack_str}

## Commandes essentielles
- Lancer les tests : `{commande_tests}`
- Démarrer l'application : `{commande_lancer}`

## Conventions de code
{conventions_str}

## Instructions pour Claude
- Toujours lancer les tests avant de confirmer qu'un fix fonctionne
- Commits séparés par fichier modifié
- Ne jamais commiter les fichiers .env ou les secrets

<important if="modifying database">
Ne jamais modifier le schéma de base de données sans créer une migration explicite.
</important>
"""

# Exemple d'utilisation
contenu = generer_claude_md(
    nom_projet="Mon API FastAPI",
    stack=["Python 3.11", "FastAPI", "PostgreSQL", "SQLAlchemy"],
    commande_tests="pytest tests/ -v",
    commande_lancer="uvicorn main:app --reload",
    conventions=[
        "Branches : feature/xxx ou fix/xxx",
        "Commits : conventional commits (feat:, fix:, docs:)",
        "PR : squash merge uniquement",
        "Pas de print() en production, utiliser logging"
    ]
)

print(contenu)
print(f"\n--- Longueur : {len(contenu.splitlines())} lignes (recommandé < 200) ---")

---
## Section 3 — Les Commandes (Slash Commands)

### Concept

Les **commandes** sont vos workflows personnalisés, invocables avec `/nom-commande`.  
Elles vivent dans `.claude/commands/` et sont versionnées avec votre projet (git).

```
.claude/
└── commands/
    ├── review-pr.md        → /review-pr
    ├── deploy.md           → /deploy
    └── debug-prod.md       → /debug-prod
```

### Structure d'une commande

Un fichier `.claude/commands/ma-commande.md` contient :

```markdown
---
name: ma-commande
description: Ce que fait la commande — utilisé pour l'autocomplétion
argument-hint: [nom-du-paramètre]
model: sonnet
allowed-tools: Bash, Read, Write
---

Instructions pour Claude quand cette commande est invoquée.

Par exemple : analyser les changements git non commités,
créer un résumé, puis proposer un message de commit.
```

### Les champs frontmatter les plus utiles

| Champ | Description | Exemple |
|-------|-------------|---------|
| `name` | Nom de la commande `/xxx` | `review-pr` |
| `description` | Affiché dans l'autocomplétion | `"Review the current PR for bugs"` |
| `argument-hint` | Indice dans l'autocomplétion | `[issue-number]` |
| `model` | Modèle forcé pour cette commande | `haiku`, `sonnet`, `opus` |
| `allowed-tools` | Outils autorisés sans confirmation | `Bash, Read, Write` |
| `context: fork` | Exécuter dans un sous-agent isolé | `fork` |
| `effort` | Niveau de réflexion du modèle | `low`, `max` |

### Commandes officielles les plus utiles (80 au total)

**Gestion de session :**
- `/compact [hint]` — Compacte le contexte (avec focus optionnel)
- `/clear` — Nouvelle conversation vide
- `/rewind` — Revenir à un point précédent (checkpoint)
- `/resume` — Reprendre une session précédente
- `/rename [nom]` — Nommer la session courante

**Modèle & Modes :**
- `/plan [description]` — Entrer en mode planification
- `/model [modèle]` — Changer de modèle à la volée
- `/effort [low|max]` — Ajuster le niveau de réflexion
- `/focus` — Masquer les appels d'outils, voir seulement le résultat

**Qualité & Debug :**
- `/review` — Review du PR courant
- `/security-review` — Analyse de sécurité des changements
- `/doctor` — Diagnostic de l'installation

> 💡 **Règle Boris Cherny** : Si vous faites quelque chose **plus d'une fois par jour**, transformez-le en commande. Exemples : `/techdebt`, `/context-dump`, `/analytics`.

In [ ]:
from pathlib import Path

# Exercice 3 : Créer des commandes personnalisées pour votre projet

commandes_exemples = {
    "commit-message": {
        "description": "Génère un message de commit conventionnel pour les changements en cours",
        "model": "haiku",
        "allowed_tools": "Bash, Read",
        "contenu": """Analyse les changements git non commités avec `git diff --staged`.

1. Identifie le type de changement : feat, fix, docs, refactor, test, chore
2. Identifie le scope (module impacté)
3. Rédige un message de commit au format : `type(scope): description`
4. Propose 3 variantes du message, du plus court au plus détaillé

Ne commit pas — affiche seulement les suggestions."""
    },
    
    "debug-session": {
        "description": "Lance un terminal en arrière-plan et surveille les logs pour déboguer",
        "model": "sonnet",
        "allowed_tools": "Bash, Read",
        "contenu": """Pour déboguer l'application :

1. Lance le serveur en arrière-plan avec `nohup ... &`
2. Surveille les logs d'erreur
3. Identifie les erreurs récurrentes
4. Propose des corrections avec contexte

Prends des screenshots si une interface web est disponible."""
    },
    
    "pr-ready": {
        "description": "Vérifie qu'une branche est prête pour une Pull Request",
        "model": "sonnet", 
        "allowed_tools": "Bash, Read",
        "contenu": """Vérifie la branche courante pour une PR :

1. `git diff main...HEAD` — résumé des changements
2. Lance les tests : `pytest tests/ -v`  
3. Vérifie le linter : `ruff check .`
4. Vérifie qu'il n'y a pas de fichiers sensibles stagés
5. Génère un template de description de PR

Rapport final : ✅ Prêt / ⚠️ À corriger / ❌ Bloquant"""
    }
}

def afficher_commande(nom: str, config: dict) -> str:
    outils = config.get("allowed_tools", "Read")
    return f"""---
name: {nom}
description: {config['description']}
model: {config['model']}
allowed-tools: {outils}
---

{config['contenu']}
"""

print("=== Exemples de commandes à créer dans .claude/commands/ ===\n")
for nom, config in commandes_exemples.items():
    print(f"📄 .claude/commands/{nom}.md")
    print("-" * 50)
    print(afficher_commande(nom, config))
    print()

---
## Section 4 — Les Skills

### Concept

Les **Skills** sont des capacités réutilisables avec un frontmatter YAML.  
Différence clé avec les commandes :
- **Commande** = workflow séquentiel, invoqué par l'utilisateur
- **Skill** = capacité/savoir-faire, invoqué automatiquement par Claude selon le contexte

```
.claude/
└── skills/
    └── mon-skill/          ← un skill = un dossier
        ├── SKILL.md        ← fichier principal obligatoire
        ├── examples.md     ← exemples (optionnel)
        ├── reference.md    ← référence technique (optionnel)
        └── scripts/        ← scripts réutilisables (optionnel)
```

### Structure d'un SKILL.md

```markdown
---
name: weather-fetcher
description: Fetches current weather data. Use when weather information is needed.
context: fork
agent: general-purpose
allowed-tools: Bash
---

## Objectif
Récupérer la météo actuelle pour une ville donnée.

## API à utiliser
Open-Meteo (gratuite, pas de clé requise) :
`https://api.open-meteo.com/v1/forecast?latitude=48.85&longitude=2.35&current_weather=true`

## Format de sortie attendu
```json
{"temperature": 18.5, "windspeed": 12.3, "city": "Paris"}
```

## Gotchas (pièges connus)
- L'API retourne la température en Celsius par défaut
- Ne pas appeler si les données < 5 min (utiliser le cache)
```

### Champs frontmatter clés

| Champ | Description | Quand l'utiliser |
|-------|-------------|-----------------|
| `description` | Déclencheur d'invocation auto | **Toujours** — c'est la clé de l'auto-discovery |
| `context: fork` | Isole dans un sous-agent | Tâches longues, side-effects possibles |
| `allowed-tools` | Outils sans confirmation | Automatiser des actions répétitives |
| `user-invocable: false` | Cache du menu `/` | Skills réservés aux agents |
| `paths` | Active sur certains fichiers | Ex: `"**/*.test.ts"` pour un skill de tests |
| `hooks` | Hooks spécifiques à ce skill | Métriques, nettoyage, notifications |

### Skills officiels bundlés (9)

| Skill | Usage |
|-------|-------|
| `/code-review` | Review le diff courant, bugs et sécurité |
| `/debug` | Débogue une commande ou du code |
| `/batch` | Exécute des commandes sur plusieurs fichiers |
| `/loop` | Relance un prompt à intervalles réguliers |
| `/run` | Lance l'app et vérifie visuellement |
| `/verify` | Vérifie qu'un changement fonctionne |
| `/fewer-permission-prompts` | Réduit les demandes de permissions |

### Conseils des créateurs (Thariq, Anthropic)

1. **La description est un déclencheur, pas un résumé** — écrivez "quand invoquer" pas "ce que ça fait"
2. **Section Gotchas obligatoire** — les pièges connus = contenu le plus précieux
3. **Ne pas être trop directif** — donnez des objectifs et contraintes, pas du pas-à-pas
4. **N'écrivez pas l'évident** — focalisez sur ce qui sort Claude de son comportement par défaut
5. **Incluez des scripts** pour que Claude compose plutôt que reconstruire du boilerplate

In [ ]:
# Exercice 4 : Comparer Commandes vs Skills — quand utiliser quoi ?

comparaison = {
    "Questions à se poser": [
        "Qui l'invoque ?",
        "Quand l'invoquer ?",
        "Isolé dans un sous-agent ?",
        "Versionné avec le projet ?",
        "Exemple typique",
    ],
    "Commande (.claude/commands/)": [
        "L'utilisateur (manuellement)",
        "Sur demande explicite : /ma-commande",
        "Optionnel (context: fork)",
        "✅ Oui (git)",
        "/review-pr, /deploy, /commit-message",
    ],
    "Skill (.claude/skills/)": [
        "Claude (automatiquement selon contexte)",
        "Quand la description matche la tâche",
        "Recommandé pour tâches longues",
        "✅ Oui (git)",
        "weather-fetcher, code-formatter, api-caller",
    ],
}

print("=" * 70)
print("COMMANDES vs SKILLS — Guide de décision")
print("=" * 70)

questions = comparaison["Questions à se poser"]
for i, question in enumerate(questions):
    cmd = comparaison["Commande (.claude/commands/)"][i]
    skill = comparaison["Skill (.claude/skills/)"][i]
    print(f"\n📌 {question}")
    print(f"   Commande : {cmd}")
    print(f"   Skill    : {skill}")

print("\n" + "=" * 70)
print("\n🎯 RÈGLE SIMPLE :")
print("  → Vous déclenchez = Commande")
print("  → Claude déclenche selon le contexte = Skill")
print("  → Les deux peuvent co-exister et se référencer")

---
## Section 5 — Les Sous-agents (Subagents)

### Concept

Les **sous-agents** sont des assistants Claude spécialisés que Claude peut déléguer automatiquement.  
Chaque sous-agent a son propre contexte, ses propres outils, et son propre modèle.

```
Claude (session principale)
    ├── Contexte principal
    └── Délègue à →
            ├── sous-agent "code-reviewer"
            │     └── Son propre contexte, lectures, analyses
            ├── sous-agent "test-runner"  
            │     └── Lance les tests en isolation
            └── sous-agent "doc-writer"
                  └── Rédige la documentation
```

### Avantage clé : la gestion du contexte

Quand un sous-agent fait 20 lectures de fichiers + 12 grep + 3 dead-ends,  
**tout ça reste dans son contexte** — seul le rapport final remonte.

### Structure d'un sous-agent

```yaml
# .claude/agents/mon-agent.md
---
name: code-reviewer
description: PROACTIVELY reviews code changes for bugs, security issues, and style
model: opus
tools: Read, Glob, Grep
permissionMode: plan
color: blue
---

Tu es un expert en code review.

Pour chaque PR ou ensemble de changements :
1. Vérifie les bugs logiques
2. Identifie les failles de sécurité (injections, XSS, secrets exposés)
3. Vérifie la couverture de tests
4. Propose des améliorations de lisibilité

Format de sortie :
- 🔴 Critique : [description]
- 🟡 Important : [description]  
- 🟢 Suggestion : [description]
```

### Champs frontmatter importants

| Champ | Description | Valeurs |
|-------|-------------|---------|
| `name` | Identifiant unique | `code-reviewer` |
| `description` | Quand Claude l'invoque | Mettre `PROACTIVELY` pour auto-invocation |
| `model` | Modèle dédié | `haiku` (rapide/cheap), `sonnet`, `opus` |
| `tools` | Outils autorisés | `Read, Write, Bash, Glob, Grep` |
| `permissionMode` | Niveau d'autonomie | `plan`, `acceptEdits`, `auto`, `bypassPermissions` |
| `maxTurns` | Limite de tours | `10` (évite les boucles infinies) |
| `skills` | Skills pré-chargés | `["weather-fetcher"]` |
| `color` | Couleur dans le terminal | `blue`, `red`, `green`... |
| `isolation` | Worktree git isolé | `"worktree"` |

### Sous-agents officiels (5)

| Agent | Modèle | Outils | Usage |
|-------|--------|--------|-------|
| `general-purpose` | inherit | Tous | Tâches complexes multi-étapes |
| `Explore` | haiku | Lecture seule | Recherche rapide dans le code |
| `Plan` | inherit | Lecture seule | Planification avant écriture |
| `statusline-setup` | sonnet | Read, Edit | Configure la barre de statut |
| `claude-code-guide` | haiku | Web+Glob+Grep | Questions sur Claude Code |

### Appeler un sous-agent depuis un agent

```
# Dans le corps d'un fichier agent .md :
Agent(subagent_type="code-reviewer", description="Review auth changes", prompt="...")

# ⚠️ NE PAS utiliser bash pour appeler un agent :
# ❌ subprocess.run(["claude", "--agent", "code-reviewer"])
# ✅ Utiliser l'outil Agent() directement
```

> 💡 **Tip Boris** : Dites juste "use subagents" pour que Claude jette plus de puissance de calcul sur un problème — les fenêtres de contexte séparées améliorent les résultats.

In [ ]:
# Exercice 5 : Concevoir une équipe de sous-agents pour un projet

def concevoir_equipe_agents(description_projet: str, besoins: list[str]) -> list[dict]:
    """Conçoit une équipe de sous-agents adaptée à un projet."""
    
    agents_disponibles = {
        "code-reviewer": {
            "model": "opus",
            "tools": "Read, Glob, Grep",
            "permissionMode": "plan",
            "color": "blue",
            "role": "Revue de code, bugs, sécurité"
        },
        "test-writer": {
            "model": "sonnet",
            "tools": "Read, Write, Edit, Bash",
            "permissionMode": "acceptEdits",
            "color": "green",
            "role": "Génération de tests unitaires et d'intégration"
        },
        "doc-writer": {
            "model": "sonnet",
            "tools": "Read, Write, Glob",
            "permissionMode": "acceptEdits",
            "color": "yellow",
            "role": "Documentation, README, commentaires"
        },
        "security-auditor": {
            "model": "opus",
            "tools": "Read, Glob, Grep",
            "permissionMode": "plan",
            "color": "red",
            "role": "Audit de sécurité, OWASP, secrets exposés"
        },
        "refactor-expert": {
            "model": "sonnet",
            "tools": "Read, Write, Edit, Bash",
            "permissionMode": "acceptEdits",
            "color": "purple",
            "role": "Refactoring, clean code, performance"
        },
    }
    
    equipe = []
    for besoin in besoins:
        if besoin in agents_disponibles:
            agent = agents_disponibles[besoin]
            equipe.append({"nom": besoin, **agent})
    
    return equipe


# Exemple : projet e-commerce
equipe = concevoir_equipe_agents(
    description_projet="API e-commerce avec paiements",
    besoins=["code-reviewer", "security-auditor", "test-writer", "doc-writer"]
)

print("=== Équipe d'agents pour votre projet ===\n")
for agent in equipe:
    print(f"🤖 {agent['nom']}")
    print(f"   Rôle    : {agent['role']}")
    print(f"   Modèle  : {agent['model']}")
    print(f"   Outils  : {agent['tools']}")
    print(f"   Mode    : {agent['permissionMode']}")
    print(f"   Couleur : {agent['color']}")
    print()

print("\n📁 Structure à créer :")
print(".claude/")
print("└── agents/")
for agent in equipe:
    print(f"    ├── {agent['nom']}.md")

---
## Section 6 — Les Settings (Configuration)

### Hiérarchie des paramètres

Claude Code applique les settings dans cet ordre de priorité (du plus fort au plus faible) :

```
1. Managed (organisation IT) → non modifiable
2. Arguments CLI             → session uniquement
3. .claude/settings.local.json  → personnel, non partagé (gitignore)
4. .claude/settings.json        → équipe, versionné (git)
5. ~/.claude/settings.json      → global personnel
```

### Paramètres essentiels

```json
// .claude/settings.json
{
  "$schema": "https://json.schemastore.org/claude-code-settings.json",
  
  // Modèle par défaut
  "model": "claude-sonnet-4-6",
  
  // Langue des réponses
  "language": "french",
  
  // Permissions (outils autorisés sans demande)
  "permissions": {
    "allow": [
      "Bash(git status)",
      "Bash(git diff *)",
      "Bash(npm run *)",
      "Read(**)",
      "Edit(src/**)"
    ],
    "deny": [
      "Bash(rm -rf *)",
      "Bash(git push --force)"
    ]
  },
  
  // Attribution dans les commits
  "attribution": {
    "commit": "",
    "coAuthor": false
  }
}
```

### Permissions : syntaxe wildcards

| Permission | Signification |
|-----------|--------------|
| `Bash(git *)` | Toutes les commandes git |
| `Bash(npm run *)` | Tous les scripts npm |
| `Read(**)` | Lecture de tous les fichiers |
| `Edit(src/**)` | Édition dans src/ uniquement |
| `Edit(/docs/**)` | Édition dans /docs/ |

### Paramètres utiles

| Clé | Type | Description |
|-----|------|-------------|
| `model` | string | Modèle par défaut (`sonnet`, `opus`, `haiku`) |
| `language` | string | Langue des réponses Claude |
| `fastMode` | boolean | Active le mode rapide (Opus avec output rapide) |
| `outputStyle` | string | `"explanatory"` pour voir le raisonnement |
| `thinkingMode` | boolean | `true` pour voir le raisonnement interne |
| `autoCompact` | boolean | Compaction automatique du contexte |
| `attribution.commit` | string | Texte co-author dans les commits (`""` pour désactiver) |

### Sandbox — réduire les prompts de permission de 84%

```json
{
  "sandbox": true
}
```

Le sandbox isole les fichiers et le réseau.  
Moins de prompts interruptifs, plus d'autonomie sécurisée.

> 💡 **Tip** : Utilisez `/fewer-permission-prompts` — ce skill scanne votre historique de session et génère automatiquement la liste des permissions à ajouter dans settings.json.

---
## Section 7 — Gestion du Contexte

### Le problème : la "Context Rot"

Plus une session dure, plus le contexte se remplit.  
Au-delà d'un certain seuil, la qualité des réponses **se dégrade**.

```
Fenêtre de contexte (1M tokens)
│
├── 0-30% ████████████░░░░░░░░░░░░░░░  Zone optimale ✅
│
├── 30-40% ████████████████░░░░░░░░░░  Attention ⚠️ (dumb zone commence)
│
├── 40-60% ████████████████████████░░  Dégradation notable
│
└── 60%+   ████████████████████████████  Zone à éviter ❌
```

**Conseil** :
- **Débutants** : garder sous 40%, wrapper à 60%
- **Experts** : agressivement sous 30%, maximum 60% pour tâches simples

### Commandes de gestion du contexte

| Commande | Usage | Quand l'utiliser |
|----------|-------|-----------------|
| `/context` | Visualise l'usage actuel | Vérifier régulièrement |
| `/compact [hint]` | Compacte avec focus | Milieu de tâche, ok d'être flou |
| `/compact focus on auth refactor` | Compacte avec focus précis | Garder le fil d'une tâche |
| `/clear` | Repart à zéro | Nouvelle tâche, contexte frais |
| `/rewind` | Revient à un checkpoint | Après une mauvaise tentative |

### Stratégies selon la situation

```
Tour terminé → Que faire ?
    │
    ├── Continuer la même tâche ?
    │       → Continue (contexte conservé)
    │
    ├── Mauvaise tentative de Claude ?
    │       → /rewind (rembobiner + re-prompter)
    │
    ├── Contexte lourd, tâche similaire ?
    │       → /compact [focus hint]
    │
    ├── Nouvelle tâche indépendante ?
    │       → /clear + brief (vous contrôlez ce qui passe)
    │
    └── Tâche nécessite beaucoup de lectures ?
            → Déléguer à un sous-agent
```

### Le /rewind — technique avancée

Au lieu de laisser "tentative ratée + correction" polluer le contexte :

```
# Session standard (polluée) :
User: Implémente l'auth
Claude: [mauvaise implémentation avec JWT]
User: Non, utilise OAuth  ← contexte pollué par la mauvaise tentative
Claude: [refait avec OAuth mais contexte lourd]

# Avec /rewind :
User: Implémente l'auth
Claude: [mauvaise implémentation avec JWT]
→ Double-Esc ou /rewind
→ Re-prompt: "Implémente l'auth avec OAuth (pas JWT)"
Claude: [implémentation propre, contexte léger]
```

> 💡 **Tip Thariq** : Avant de rembobiner, demandez à Claude "summarize from here" — il écrit un message de passation de soi à lui-même pour la prochaine session.

---
## Section 8 — Workflows de Développement

### Le pattern universel

Tous les grands workflows convergent vers la même architecture :

```
Research → Plan → Execute → Review → Ship
```

### Workflow recommandé étape par étape

#### Étape 1 : Spécifier
```
"Crée une API REST pour gérer des todos avec :
 - CRUD complet
 - Auth JWT
 - Tests unitaires
 Interviews-moi avec AskUserQuestion pour clarifier les ambiguïtés"
```

#### Étape 2 : Planifier (`/plan`)
```
/plan implémenter l'API todos avec auth JWT
→ Claude explore le codebase existant
→ Propose une architecture détaillée
→ Liste les fichiers à créer/modifier
→ Vous validez avant tout changement
```

#### Étape 3 : Exécuter (avec sous-agents)
```
"Implémente le plan validé.
 Use subagents pour les tâches parallèles :
 - Un agent pour le backend
 - Un agent pour les tests
 - Un agent pour la documentation"
```

#### Étape 4 : Review
```
/review           → Review locale
/security-review  → Audit sécurité
/code-review      → Review approfondie multi-agents
```

#### Étape 5 : Ship
```
/pr-ready         → Votre commande custom (Section 3)
→ Tests passent ✅
→ Linter OK ✅  
→ PR créée avec description générée
```

### Mode Plan vs Mode Normal

| | Mode Normal | Mode Plan (`/plan`) |
|--|------------|-------------------|
| **Écriture de code** | Oui immédiatement | Non (lecture seule) |
| **Lecture de fichiers** | Oui | Oui |
| **Propose avant d'agir** | Pas systématiquement | Toujours |
| **Usage recommandé** | Tâches simples/claires | Tâches complexes, nouvelles features |

### Modèles : quand utiliser quoi ?

| Modèle | Points forts | Usage recommandé |
|--------|-------------|-----------------|
| **Haiku** | Rapide, économique | Exploration, recherche simple, tâches répétitives |
| **Sonnet** | Équilibré | Code du quotidien, refactoring, tests |
| **Opus** | Maximum d'intelligence | Architecture, débogage complexe, mode Plan |

> 💡 **Tip Cat Wu (Anthropic)** : Utilisez Opus pour le mode Plan, Sonnet pour le code — le meilleur des deux mondes.

### Git Best Practices avec Claude Code

```bash
# ✅ PRs petites et focalisées (p50 = 118 lignes selon Boris)
git checkout -b feature/add-auth-endpoint

# ✅ Committer souvent (au moins 1x/heure)
git commit -m "feat(auth): add JWT token generation"

# ✅ Squash merge pour un historique propre
git merge --squash feature/add-auth-endpoint

# ✅ Utiliser /code-review avant de merger
/review
```

---
## Section 9 — Tips & Tricks (Sélection des 83 conseils)

### 🎯 Prompting (ne micro-managez pas)

| Conseil | Source |
|---------|--------|
| "Fix" suffit — collez le bug, dites "fix", laissez Claude choisir comment | Boris Cherny |
| Après un fix médiocre : "Knowing everything you know now, scrap this and implement the elegant solution" | Boris Cherny |
| Challengez Claude : "Grill me on these changes and don't make a PR until I pass your test" | Boris Cherny |
| Utilisez `ultrathink` dans vos prompts pour déclencher un raisonnement plus profond | Docs officiels |

### 📋 Planning

| Conseil | Source |
|---------|--------|
| Toujours commencer avec `/plan` pour les tâches complexes | Boris Cherny |
| Commencez par un spec minimal, demandez à Claude de vous interviewer, puis nouvelle session pour exécuter | Thariq (Anthropic) |
| Planification par tranches verticales (DB + service + UI ensemble) plutôt qu'horizontales (DB phase, puis API phase...) | Matt Pocock |
| Spinup un second Claude pour reviewer votre plan en tant que "staff engineer" | Boris Cherny |

### 🧠 Contexte

| Conseil | Source |
|---------|--------|
| Dumb zone : ~40% du contexte. Garder sous 30% pour les experts | Dex Horthy |
| `/compact` avec hint bat l'autocompact — vous gardez le contrôle | Thariq |
| Utilisez des sous-agents pour les recherches longues — les dead-ends restent dans leur contexte | Thariq |
| Rewind > corriger — revenez en arrière plutôt que d'accumuler les corrections | Thariq |

### 🔄 Session

| Conseil | Source |
|---------|--------|
| Nouvelle tâche = nouvelle session | Thariq |
| `/rename` les sessions importantes, `/resume` plus tard | Cat Wu (Anthropic) |
| Utilisez les recaps pour les longues sessions | Boris Cherny |

### ⚙️ Workflows Avancés

| Conseil | Source |
|---------|--------|
| `/loop` pour tâches locales récurrentes (jusqu'à 7 jours) | Boris Cherny |
| `/schedule` pour tâches cloud récurrentes (même machine éteinte) | Boris Cherny |
| `/permissions` avec wildcards plutôt que `dangerously-skip-permissions` | Boris Cherny |
| Mode `auto` (Shift+Tab) au lieu de `dangerously-skip-permissions` — classifier IA pour décider | Boris Cherny |
| ASCII diagrams pour comprendre votre architecture | Boris Cherny |

### 🐛 Debugging

| Conseil | Source |
|---------|--------|
| Prenez des screenshots et partagez-les à Claude quand vous êtes bloqué | Shayan |
| Lancez le terminal en arrière-plan pour un meilleur débogage des logs | Shayan |
| Utilisez un MCP browser (Claude in Chrome, Playwright) pour que Claude voit les console logs | Docs officiels |
| Agentic search (glob + grep) bat le RAG — Claude Code a essayé et abandonné les bases vectorielles | Boris Cherny |

### 🛠️ CLAUDE.md

| Conseil | Source |
|---------|--------|
| Sous 200 lignes par fichier | Boris Cherny |
| `<important if="...">` pour les règles critiques | Dex Horthy |
| N'importe qui doit pouvoir lancer "run the tests" depuis une session fraîche | Dex Horthy |
| `settings.json` pour les comportements déterministes (attribution, permissions) — pas CLAUDE.md | davila7 |

---
## Section 10 — Architecture Command → Agent → Skill

### Le pattern d'orchestration

C'est le pattern le plus puissant de Claude Code, illustré par le projet `weather` du repo :

```
Utilisateur
    │
    │  /weather-orchestrator      ← Commande (point d'entrée)
    ▼
.claude/commands/weather-orchestrator.md
    │
    │  Invoke Agent("weather-agent", ...)
    ▼
.claude/agents/weather-agent.md  ← Agent (orchestrateur)
    │  (avec weather-fetcher préchargé dans skills:)
    │
    ├── Utilise weather-fetcher skill (préchargé)
    │       └── Appelle Open-Meteo API → retourne température
    │
    └── Skill Tool("weather-svg-creator")
            └── Génère le SVG de la météo
                    └── Écrit orchestration-workflow/weather.svg
```

### Deux types de skills dans ce pattern

| Type | Mécanisme | Quand l'utiliser |
|------|-----------|-----------------|
| **Agent skill** (préchargé) | `skills:` dans le frontmatter de l'agent | Capacités toujours disponibles pour cet agent |
| **Skill tool** (invoqué) | `Skill("nom-skill")` dans le prompt | Capacités spécialisées appelées à la demande |

### Exemple concret : un orchestrateur de déploiement

```markdown
# .claude/commands/deploy.md
---
name: deploy
description: Deploy the current branch to staging or production
argument-hint: [staging|production]
model: sonnet
---

Orchestre le déploiement de la branche courante.

1. Lance l'agent "pre-deploy-checker" pour vérifier les prérequis
2. Si staging : délègue à "staging-deployer"
3. Si production : délègue à "production-deployer" (mode plan obligatoire)
4. Après déploiement : lance le skill "smoke-tester"
5. En cas d'échec : lance le skill "rollback-handler"
```

```markdown
# .claude/agents/pre-deploy-checker.md
---
name: pre-deploy-checker
description: Checks all prerequisites before a deployment
model: haiku
tools: Bash, Read
skills: ["ci-status-checker", "security-scanner"]
maxTurns: 10
---

Vérifie avant tout déploiement :
1. Tests CI passent (skill ci-status-checker)
2. Aucune vulnérabilité critique (skill security-scanner)
3. Variables d'environnement configurées
4. Retourne : ✅ OK / ❌ BLOQUANT avec détails
```

### Structure de projet complète recommandée

```
mon-projet/
├── CLAUDE.md                    ← Mémoire projet (< 200 lignes)
├── .claude/
│   ├── settings.json            ← Config équipe
│   ├── settings.local.json      ← Config personnelle (gitignore)
│   ├── commands/
│   │   ├── deploy.md            → /deploy
│   │   ├── review-pr.md         → /review-pr
│   │   └── commit-message.md    → /commit-message
│   ├── agents/
│   │   ├── code-reviewer.md
│   │   ├── security-auditor.md
│   │   └── test-writer.md
│   ├── skills/
│   │   ├── api-caller/
│   │   │   └── SKILL.md
│   │   └── smoke-tester/
│   │       ├── SKILL.md
│   │       └── scripts/run-smoke.sh
│   └── hooks/
│       └── hooks.py             ← Notifications, auto-format
└── src/
    └── ...
```

In [ ]:
import json

# Exercice final : Générateur de structure .claude/ complète
def generer_structure_claude(
    projet: str,
    stack: str,
    commandes_voulues: list[str],
    agents_voulus: list[str],
) -> dict:
    """Génère la structure .claude/ adaptée à votre projet."""
    
    settings = {
        "$schema": "https://json.schemastore.org/claude-code-settings.json",
        "model": "claude-sonnet-4-6",
        "language": "french",
        "permissions": {
            "allow": [
                "Bash(git status)",
                "Bash(git diff *)",
                "Bash(git log *)",
                "Read(**)",
            ],
            "deny": [
                "Bash(rm -rf *)",
                "Bash(git push --force *)"
            ]
        },
        "attribution": {
            "commit": "",
            "coAuthor": False
        }
    }
    
    fichiers_a_creer = []
    
    # CLAUDE.md
    fichiers_a_creer.append({
        "chemin": "CLAUDE.md",
        "contenu": f"# {projet}\n\n## Stack\n{stack}\n\n## Tests\n- [À compléter]\n\n## Conventions\n- [À compléter]"
    })
    
    # Settings
    fichiers_a_creer.append({
        "chemin": ".claude/settings.json",
        "contenu": json.dumps(settings, indent=2, ensure_ascii=False)
    })
    
    # Commandes
    for cmd in commandes_voulues:
        fichiers_a_creer.append({
            "chemin": f".claude/commands/{cmd}.md",
            "contenu": f"---\nname: {cmd}\ndescription: [À compléter]\nmodel: sonnet\n---\n\n[Instructions pour la commande /{cmd}]"
        })
    
    # Agents
    for agent in agents_voulus:
        fichiers_a_creer.append({
            "chemin": f".claude/agents/{agent}.md",
            "contenu": f"---\nname: {agent}\ndescription: [À compléter]\nmodel: sonnet\ntools: Read, Glob, Grep\n---\n\n[Instructions pour l'agent {agent}]"
        })
    
    return {
        "projet": projet,
        "fichiers": fichiers_a_creer,
        "nb_fichiers": len(fichiers_a_creer)
    }


# Générer pour un projet exemple
structure = generer_structure_claude(
    projet="Mon API Python",
    stack="Python 3.11, FastAPI, PostgreSQL, pytest",
    commandes_voulues=["review-pr", "commit-message", "deploy-staging"],
    agents_voulus=["code-reviewer", "test-writer", "doc-writer"],
)

print(f"=== Structure .claude/ pour '{structure['projet']}' ===\n")
print(f"📁 {structure['nb_fichiers']} fichiers à créer :\n")

for fichier in structure["fichiers"]:
    print(f"  📄 {fichier['chemin']}")

print("\n=== Contenu de settings.json ===")
for fichier in structure["fichiers"]:
    if fichier["chemin"] == ".claude/settings.json":
        print(fichier["contenu"])
        break

print("\n✅ Copiez ces fichiers dans votre projet pour démarrer !")
print("💡 Complétez les [À compléter] avec vos instructions spécifiques.")

---
## Récapitulatif & Checklist de démarrage

### Ce que vous avez appris

| Concept | Ce qu'il fait | Où le mettre |
|---------|--------------|-------------|
| **CLAUDE.md** | Mémoire persistante du projet | Racine du projet |
| **Commandes** | Workflows invocables via `/nom` | `.claude/commands/` |
| **Skills** | Capacités auto-découvertes par Claude | `.claude/skills/nom/SKILL.md` |
| **Sous-agents** | Délégation de tâches avec contexte isolé | `.claude/agents/` |
| **Settings** | Configuration équipe/personnelle | `.claude/settings.json` |
| **Contexte** | Gérer la "dumb zone" avec `/compact`, `/rewind` | Commandes de session |

### Checklist pour votre premier projet

```
□ 1. Installer Claude Code : npm install -g @anthropic-ai/claude-code
□ 2. Vérifier : claude /doctor
□ 3. Créer CLAUDE.md (< 200 lignes, commandes de test incluses)
□ 4. Créer .claude/settings.json avec les permissions de base
□ 5. Créer 2-3 commandes pour vos workflows répétitifs
□ 6. Lancer avec /plan pour les nouvelles features
□ 7. Garder le contexte sous 40% (/context pour vérifier)
□ 8. Committer souvent (au moins 1x/heure)
□ 9. Utiliser /review avant chaque PR
□ 10. Mettre à jour Claude Code chaque jour
```

### Ressources pour aller plus loin

- **Repo source** : [claude-code-best-practice](https://github.com/shanraisshan/claude-code-best-practice)
- **Docs officielles** : [code.claude.com/docs](https://code.claude.com/docs)
- **Skills officiels** : [github.com/anthropics/skills](https://github.com/anthropics/skills)
- **Community** : [r/ClaudeCode](https://www.reddit.com/r/ClaudeCode/)
- **Boris Cherny** (créateur) : [@bcherny](https://x.com/bcherny) sur X
- **Thariq** (Anthropic) : [@trq212](https://x.com/trq212) sur X

### Vidéos recommandées

| Vidéo | Durée | Pour qui |
|-------|-------|---------|
| [Inside Claude Code (Boris @ Y Combinator)](https://youtu.be/PQU9o_5rHC4) | ~45min | Tous niveaux |
| [Building Claude Code (The Pragmatic Engineer)](https://youtu.be/julbw1JuAz0) | ~60min | Développeurs |
| [From Vibe Coding to Agentic Engineering (Andrej Karpathy)](https://www.youtube.com/watch?v=96jN2OCOfLs) | ~45min | Avancés |

---

*Notebook créé à partir de [claude-code-best-practice](https://github.com/shanraisshan/claude-code-best-practice) — Mai 2026*